In [ ]:
import os
import json
import re
from typing import List, Dict
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pdfplumber
from datetime import datetime

# Define version constant
POLARBRIEF_VERSION = "PolarBrief v1.0"

# === CONFIGURATION ===
pdf_path = "data.pdf"
poppler_path = r"C:\Users\ayush\Videos\poppler-24.08.0\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# === TEXT CLEANING UTILITIES ===
def clean_text(text: str) -> str:
    text = re.sub(r"[•·●♦▪•∙]", "", text)
    text = re.sub(r"[^\w\s,.:;()\"'-]", "", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

def is_noisy(text: str, threshold: float = 0.6) -> bool:
    if not text:
        return True
    non_alpha = sum(1 for c in text if not c.isalnum())
    return (non_alpha / len(text)) > threshold

def has_repeated_characters(text: str, repeat_threshold: int = 4) -> bool:
    return bool(re.search(r'(.)\1{' + str(repeat_threshold) + ',}', text))

# === MAIN FUNCTION ===
def extract_text_with_fallback(pdf_path: str) -> List[Dict]:
    final_output = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)

            for page_num, page in enumerate(pdf.pages, start=1):
                print(f"📄 Page {page_num}/{total_pages}: Trying PDF text extraction...")

                page_line_no = 1
                text = page.extract_text()
                lines = text.split("\n") if text else []

                if lines and sum(len(l.strip()) for l in lines) > 20:
                    for line in lines:
                        cleaned = line.strip()
                        if cleaned:
                            final_output.append({
                                "text": cleaned,
                                "page_no": f"[p{page_num} {page_line_no}]",
                                "method": "pdfplumber"
                            })
                            page_line_no += 1
                    continue  # success, skip OCR fallback

                # === OCR Fallback ===
                print(f"⚠️ Page {page_num} has no usable text — fallback to OCR")

                images = convert_from_path(pdf_path, dpi=300, first_page=page_num, last_page=page_num, poppler_path=poppler_path)
                img = images[0]
                gray = img.convert("L")
                bw = gray.point(lambda x: 0 if x < 180 else 255, '1')

                ocr_text = pytesseract.image_to_string(bw, lang='eng')
                lines = ocr_text.strip().split("\n")
                line_no = 1

                for line in lines:
                    cleaned_line = clean_text(line)
                    if (
                        cleaned_line and
                        not is_noisy(cleaned_line) and
                        not has_repeated_characters(cleaned_line)
                    ):
                        final_output.append({
                            "text": cleaned_line,
                            "page_no": f"[p{page_num} {line_no}]",
                            "method": "ocr"
                        })
                        line_no += 1

    except Exception as e:
        print(f"[ERROR] Failed during PDF processing: {e}")

    return final_output

# === RUN AND SAVE ===
output = extract_text_with_fallback(pdf_path)

with open("smart_output.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("✅ Extraction complete. Output saved to 'smart_output.json'")


📄 Page 1/26: Trying PDF text extraction...
📄 Page 2/26: Trying PDF text extraction...
📄 Page 3/26: Trying PDF text extraction...
📄 Page 4/26: Trying PDF text extraction...
📄 Page 5/26: Trying PDF text extraction...
📄 Page 6/26: Trying PDF text extraction...
📄 Page 7/26: Trying PDF text extraction...
📄 Page 8/26: Trying PDF text extraction...
📄 Page 9/26: Trying PDF text extraction...
📄 Page 10/26: Trying PDF text extraction...
📄 Page 11/26: Trying PDF text extraction...
📄 Page 12/26: Trying PDF text extraction...
📄 Page 13/26: Trying PDF text extraction...
📄 Page 14/26: Trying PDF text extraction...
📄 Page 15/26: Trying PDF text extraction...
📄 Page 16/26: Trying PDF text extraction...
📄 Page 17/26: Trying PDF text extraction...
📄 Page 18/26: Trying PDF text extraction...
📄 Page 19/26: Trying PDF text extraction...
📄 Page 20/26: Trying PDF text extraction...
📄 Page 21/26: Trying PDF text extraction...
📄 Page 22/26: Trying PDF text extraction...
📄 Page 23/26: Trying PDF text extraction.

In [2]:
def count_tokens_simple(text: str) -> int:
    return len(text.split())
def chunk_lines_simple_tokenizer(lines: List[Dict], max_tokens: int = 250) -> List[Dict]:
    chunks = []
    current_lines = []
    current_tokens = 0
    citation_line = None
    citation_page = None

    for line in lines:
        text = line["text"].strip()
        tokens = count_tokens_simple(text)

        # If adding this line exceeds limit, flush the chunk
        if current_tokens + tokens > max_tokens and current_lines:
            chunk_text = "\n".join([l["text"] for l in current_lines])
            chunks.append({
                "page": citation_page,
                "citation": citation_line,
                "text": chunk_text
            })
            current_lines = []
            current_tokens = 0
            citation_line = None
            citation_page = None

        # Start a new chunk if needed
        if not current_lines and text:
            citation_line = text
            citation_page = line["page_no"]

        current_lines.append(line)
        current_tokens += tokens

    # Final flush
    if current_lines:
        chunk_text = "\n".join([l["text"] for l in current_lines])
        chunks.append({
            "page": citation_page,
            "citation": citation_line,
            "text": chunk_text
        })

    return chunks




chunks = chunk_lines_simple_tokenizer(output, max_tokens=250)
    
with open("chunked_paragraphs.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=4)

    
print(json.dumps(chunks[:3], indent=2))

[
  {
    "page": "[p1 1]",
    "citation": "Case 2:22-cv-00223-Z Document 100 Filed 02/13/23 Page 1 of 26 PageID 3760",
    "text": "Case 2:22-cv-00223-Z Document 100 Filed 02/13/23 Page 1 of 26 PageID 3760\nUNITED STATES DISTRICT COURT\nNORTHERN DISTRICT OF TEXAS\nAMARILLO DIVISION\nALLIANCE FOR HIPPOCRATIC\nMEDICINE, et al.,\nPlaintiffs,\nv. Case No. 2:22-cv-00223-Z\nU.S. FOOD AND DRUG\nADMINISTRATION, et al.,\nDefendants.\nAMICUS CURIAE BRIEF OF MISSISSIPPI, ALABAMA, ALASKA,\nARKANSAS, FLORIDA, GEORGIA, IDAHO, INDIANA, IOWA, KANSAS,\nKENTUCKY, LOUISIANA, MONTANA, NEBRASKA, OHIO, OKLAHOMA,\nSOUTH CAROLINA, SOUTH DAKOTA, TENNESSEE, TEXAS, UTAH, AND\nWYOMING IN SUPPORT OF PLAINTIFFS\u2019 MOTION\nFOR PRELIMINARY INJUNCTION\nCase 2:22-cv-00223-Z Document 100 Filed 02/13/23 Page 2 of 26 PageID 3761\nTABLE OF CONTENTS\nPage\nTABLE OF AUTHORITIES .......................................................................................... ii\nINTRODUCTION, INTEREST OF AMICI CURIAE,\nAND SUMM

In [3]:
import json
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

# Download required resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# === Step 1: Clean OCR artifacts like dots, 000cccc, etc. ===
def clean_ocr_artifacts(text: str) -> str:
    if not text:
        return ""
    # Remove leader lines like ............ or --------
    text = re.sub(r'[\.\-]{3,}', ' ', text)

    # Remove long trailing character garbage like "000ccccceeeees"
    text = re.sub(r'[a-zA-Z0-9]{3,}[a-zA-Z0-9\s]{0,}$', '', text)

    # Remove repeated characters like "eeeeee", "cccc"
    text = re.sub(r'([a-zA-Z0-9])\1{3,}', '', text)

    # Replace multiple spaces with one
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text

# === Step 2: NLP preprocess (tokenize, lower, remove stopwords, stem) ===
def preprocess_text(text: str) -> str:
    text = clean_ocr_artifacts(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # Keep only a-z and spaces
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(lemmatizer.lemmatize(word)) for word in tokens]
    return ' '.join(tokens)


# === Step 4: Load JSON data ===
with open("chunked_paragraphs.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# === Step 5: Apply cleaning to each entry ===
for entry in chunks:
    if "chunk" in entry:
        entry["chunk"] = preprocess_text(entry["chunk"])
    if "text" in entry:  # If present (like in your raw OCR)
        entry["text"] = preprocess_text(entry["text"])



# === Step 6: Save cleaned JSON ===
with open("your_dataset_preprocessed.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

# === Preview cleaned output ===
print(json.dumps(chunks[:1], indent=2))


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


[
  {
    "page": "[p1 1]",
    "citation": "Case 2:22-cv-00223-Z Document 100 Filed 02/13/23 Page 1 of 26 PageID 3760",
    "text": "case cv z document file page pageid unit state district court northern district texa amarillo divis allianc hippocrat medicin et al plaintiff v case cv z u food drug administr et al defend amicu curia brief mississippi alabama alaska arkansa florida georgia idaho indiana iowa kansa kentucki louisiana montana nebraska ohio oklahoma south carolina south dakota tennesse texa utah wyom support plaintiff motion preliminari injunct case cv z document file page pageid tabl content page tabl author ii introduct interest amici curia summari argument background argument public interest equiti support injunct relief fda action mifepriston public interest equiti weigh strongli fda action action defi feder law b fda action undermin public interest determin state feder agenc entitl make c fda action harm public interest undermin state abil protect citizen forc state d

In [ ]:
import os
import json
import re
import numpy as np
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain_groq import ChatGroq 
from langchain.schema import HumanMessage, SystemMessage
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# === Set Groq API Key ===
os.environ["GROQ_API_KEY"] = ""

# === Initialize LangChain ChatGroq ===
llm = ChatGroq(
    model_name="llama3-8b-8192",
    temperature=0,
)

# === Pydantic Response Model ===
class ArgumentAnalysis(BaseModel):
    heading: str
    contains_argument: str
    summary: str
    polarity: str = Field(default="N/A")
    score: float = Field(ge=0, le=100)

# === LLM Prompt Function ===
def get_argument_analysis(text: str) -> ArgumentAnalysis:
    system_prompt = """You are a legal assistant AI.

Given the paragraph below from a legal brief:

1. Please summarize the paragraph <<<
    Your summary must:
            1. Do not hallicunate Your summary must be based **only** on paragraph. Do not add, infer, or interpret anything beyond what is explicitly stated.
            2. Clearly and accurately capture the **main legal points or facts** in the text in points and then summarize those points in paragraph.
            3. the summary paragraph must be of >75 words .
            4. After writing the summary, **verify** that it fully aligns with the original text and does not introduce errors or hallucinations.>>>
            
2. Give a appropriate heading that the text is about.
3. Does it contain a legal argument? (yes/no)
4. if yes , Classify it as Pro (supports Plaintiffs) or Con (supports Defendants) , else "N/A"
5. Score the argument on a scale of 0 - 100 based on weight, clarity, relevance, and quality.
6. Do not fabricate content; if uncertain, answer contains_argument: "no"

 

Respond ONLY in Valid JSON , no prose:
{
  "heading" : "..." ,
  "contains_argument": "...",
  "summary": "...",
  "polarity": "Pro/Con",
  "score": <float 0-100>
}"""
    prompt = f'<<<Paragraph:\n"""{text}""">>>'

    response = llm([
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt)
    ])
    
    try:
        match = re.search(r"\{[\s\S]+?\}", response.content.strip())
        if match:
            parsed = json.loads(match.group(0))
            return ArgumentAnalysis(**parsed)
    except (ValidationError, json.JSONDecodeError) as e:
        print("Error parsing LLM JSON:", e)
    
    return ArgumentAnalysis(
        heading="",
        contains_argument="error",
        summary=text,
        polarity="N/A",
        score=0
    )

# === Normalization Function ===
def normalize(arr: List[float]):
    arr = np.array(arr)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

final_output = []
texts = []
llm_scores = []

for chunk in chunks:
    parsed = get_argument_analysis(chunk["text"])
    
    llm_scores.append(parsed.score)
    texts.append(chunk["text"])

    def truncate_to_75_words(text):
        words = text.split()
        return ' '.join(words[:75]) + ('...' if len(words) > 75 else '')

    final_output.append({
        "page": chunk["page"],
        "citation": chunk["citation"],
        "text": chunk["text"],
        "heading": parsed.heading,
        "contains_argument": parsed.contains_argument,
        "summary": truncate_to_75_words(parsed.summary),
        "polarity": parsed.polarity,
        "source": POLARBRIEF_VERSION,
        "timestamp": datetime.now().isoformat()
    })


# === Step 2: TF-IDF Centrality ===
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(texts)
centrality_scores = cosine_similarity(tfidf_matrix, tfidf_matrix).mean(axis=1)

# === Step 3: Combine Scores ===
llm_scores_norm = normalize(llm_scores)
centrality_scores_norm = normalize(centrality_scores)
combined_scores = 0.6 * llm_scores_norm + 0.4 * centrality_scores_norm

# === Step 4: Add final_score ===
for i, item in enumerate(final_output):
    item["final_score"] = round(combined_scores[i] * 100, 2)





C:\Users\ayush\AppData\Local\Temp\ipykernel_22068\355710207.py:60: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm([


In [5]:
# === Step 5: Sort by final_score ===
final_output_sorted = sorted(final_output, key=lambda x: x["final_score"], reverse=True)
final_output_sorted = [item for item in final_output_sorted if item["contains_argument"].lower() == "yes"]

# Take the top 10 (or fewer if less than 10 arguments exist)
final_output_sorted = final_output_sorted[:10]

# === Optional: Save to file ===
with open("legal_argument_analysis_ranked.json", "w") as f:
    json.dump(final_output, f, indent=2)
with open("top_10.json", "w") as f:
    json.dump(final_output_sorted[:10], f, indent=2)

print("✅ Analysis complete and saved.")

✅ Analysis complete and saved.


In [6]:
import json

# Load your full JSON (replace filename with your actual file)
with open("legal_argument_analysis_ranked.json", "r") as f:
    data = json.load(f)

# Extract only required fields
minimal_data = [
    {
        "page": item.get("page", ""),
        "citation": item.get("citation", ""),
        "heading": item.get("heading", "")
    }
    for item in data
]

# Save the extracted data into a new JSON file
with open("legal_argument_minimal.json", "w") as f:
    json.dump(minimal_data, f, indent=2)


In [7]:
import json
from fpdf import FPDF
import unicodedata

# === Function to remove unsupported characters ===
def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    return unicodedata.normalize("NFKD", text).encode("latin1", "ignore").decode("latin1")

# === PDF Layout === 
class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 14)
        self.ln(5)

    def chapter_body(self, entry, selected_fields):
        self.set_font("Arial", "", 11)
        for field in selected_fields:
            label = clean_text(field.replace("_", " ").title())
            value = clean_text(entry.get(field, ""))
            self.multi_cell(0, 8, f"{label}: {value}")
            self.ln(1)
        self.ln(3)
        self.cell(0, 0, "-" * 80)
        self.ln(5)

# === Export Function ===
def pdfexport(json_filename, selected_fields, pdf_filename):
    with open(json_filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    pdf = PDF()
    pdf.add_page()

    for item in data:
        pdf.chapter_body(item, selected_fields)

    pdf.output(pdf_filename)
    print(f"✅ PDF saved as: {pdf_filename}")

# === Usage ===
selected_fields = ["page", "citation", "heading"]
pdfexport("legal_argument_minimal.json", selected_fields, "legal_argument_minimal.pdf")

selected_fields = ["page", "citation", "heading", "summary", "polarity"]
pdfexport("legal_argument_analysis_ranked.json", selected_fields, "legal_argument_analysis_ranked.pdf")


✅ PDF saved as: legal_argument_minimal.pdf
✅ PDF saved as: legal_argument_analysis_ranked.pdf


In [57]:
pip install FPDF

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\ayush\Videos\docu3u\myvenv\Scripts\python.exe -m pip install --upgrade pip' command.


In [8]:
import json

# === Load data ===
with open("legal_argument_analysis_ranked.json", "r", encoding="utf-8") as f:
    llm_results = json.load(f)

with open("smart_output.json", "r", encoding="utf-8") as f:
    source_lines = json.load(f)

# === Build set of valid page references ===
valid_pages = {entry.get("page_no") for entry in source_lines if entry.get("page_no")}

# === Check each LLM entry by its page field ===
invalid_entries = []
for entry in llm_results:
    if entry.get("page") not in valid_pages:
        invalid_entries.append(entry)

# === Print report ===
print(f"Total entries checked: {len(llm_results)}")
print(f"Invalid page matches: {len(invalid_entries)}")

if invalid_entries:
    print("Sample invalid entries (mismatched pages):")
    for e in invalid_entries[:5]:
        print("-", e.get("page"))




Total entries checked: 24
Invalid page matches: 0


In [9]:
import json
import random
from IPython.display import display, HTML

# === Load Summary Data (Ranked output with summaries) ===
with open("legal_argument_analysis_ranked.json", "r", encoding="utf-8") as f:
    summary_data = json.load(f)

# === Load Source Texts (original full text per entry) ===
with open("chunked_paragraphs.json", "r", encoding="utf-8") as f:
    source_data = json.load(f)

key_field = "page"  # Change this based on your actual structure
source_lookup = {entry[key_field]: entry["text"] for entry in source_data if key_field in entry}

# === Random Sample from summary data ===
sample = random.sample(summary_data, min(len(summary_data), 30))

# === Scrollable Display ===
def show_scrollable(text, label="Text"):
    display(HTML(f"<b>{label}:</b><div style='max-height:200px; overflow:auto; border:1px solid #ccc; padding:10px'>{text}</div>"))

# === Manual Review Loop ===
print("\n=== MANUAL VALIDATION START ===")
relevant_count = 0
missing_count = 0

for i, entry in enumerate(sample, 1):
    source_key = entry.get(key_field)
    original_text = source_lookup.get(source_key)

    print(f"\n--- SAMPLE {i} ---")
    if original_text:
        show_scrollable(original_text, "Original")
    else:
        print(f"[MISSING TEXT for key: {source_key}]")
        missing_count += 1
        continue

    show_scrollable(entry["summary"], "Summary")
    answer = input("Is this summary relevant to the excerpt? (y/n): ").strip().lower()
    if answer == 'y':
        relevant_count += 1

relevance_at_10 = (relevant_count / (len(sample) - missing_count)) * 100 if (len(sample) - missing_count) > 0 else 0
print(f"\n Relevance Score: {relevance_at_10:.2f}% based on {len(sample) - missing_count} samples.")




=== MANUAL VALIDATION START ===

--- SAMPLE 1 ---



--- SAMPLE 2 ---



--- SAMPLE 3 ---



--- SAMPLE 4 ---



--- SAMPLE 5 ---


...


--- SAMPLE 21 ---



--- SAMPLE 22 ---



--- SAMPLE 23 ---



--- SAMPLE 24 ---



 Relevance Score: 100.00% based on 24 samples.
